In [1]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader as DL
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF


def breaker():
    print("\n" + 50 * "-" + "\n")


def head(x, no_of_ele=5):
    print(x[:no_of_ele])


def getImages(file_path=None, file_names=None, size=None):
    images = []
    for name in file_names:
        img_path = os.path.join(file_path, f"{name}.jpg")
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            image = np.zeros((size, size), dtype=np.uint8)
        if size:
            image = cv2.resize(
                image, dsize=(size, size), interpolation=cv2.INTER_LANCZOS4
            )
        images.append(image.reshape(size, size, 1))
    return np.array(images)




In [2]:
start_time = time()

ss = pd.read_csv(
    "../input/ranzcr-clip-catheter-line-classification/sample_submission.csv"
)

ts_img_names = ss["StudyInstanceUID"].values
ts_images = getImages(
    "../input/ranzcr-clip-catheter-line-classification/test/", ts_img_names, size=384
)

breaker()
print("Time Taken to read data : {:.2f} minutes".format((time() - start_time) / 60))
breaker()





--------------------------------------------------

Time Taken to read data : 0.80 minutes

--------------------------------------------------



In [3]:
class DS(Dataset):
    def __init__(self, X=None, y=None, transform=None, mode="train"):
        self.mode = mode
        self.transform = transform
        self.X = X
        if mode == "train":
            self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        img = self.transform(self.X[idx])
        if self.mode == "train":
            return img, torch.FloatTensor(self.y[idx])
        else:
            return img




In [4]:
class CFG:
    tr_batch_size = 64
    ts_batch_size = 64

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    in_channels = 1
    OL = 11  # number of output logits (matches training labels)

    def __init__(
        self, filter_sizes=[64, 128, 256, 512], HL=[2048], epochs=50, n_folds=5
    ):
        self.filter_sizes = filter_sizes
        self.HL = HL
        self.epochs = epochs
        self.n_folds = n_folds




In [5]:
class CNN(nn.Module):
    def __init__(
        self, in_channels=1, filter_sizes=None, HL=None, OL=None, use_DP=True, DP=0.50
    ):
        super(CNN, self).__init__()
        self.use_DP = use_DP
        self.DP_ = nn.Dropout(p=DP)
        self.MP_ = nn.MaxPool2d(kernel_size=2)

        self.CN1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=filter_sizes[0],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.BN1 = nn.BatchNorm2d(num_features=filter_sizes[0], eps=1e-5)

        self.CN2 = nn.Conv2d(
            in_channels=filter_sizes[0],
            out_channels=filter_sizes[1],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.BN2 = nn.BatchNorm2d(num_features=filter_sizes[1], eps=1e-5)

        self.CN3 = nn.Conv2d(
            in_channels=filter_sizes[1],
            out_channels=filter_sizes[2],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.BN3 = nn.BatchNorm2d(num_features=filter_sizes[2], eps=1e-5)

        self.CN4 = nn.Conv2d(
            in_channels=filter_sizes[2],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.BN4 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        self.CN5 = nn.Conv2d(
            in_channels=filter_sizes[3],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.BN5 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        self.CN6 = nn.Conv2d(
            in_channels=filter_sizes[3],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.BN6 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        self.CN7 = nn.Conv2d(
            in_channels=filter_sizes[3],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.BN7 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        self.FC1 = nn.Linear(in_features=filter_sizes[3] * 3 * 3, out_features=HL[0])
        self.FC2 = nn.Linear(in_features=HL[0], out_features=OL)

    def getOptimizer(self, lr=1e-3, wd=0):
        return optim.Adam(self.parameters(), lr=lr, weight_decay=wd)

    def forward(self, x):
        x = F.relu(self.MP_(self.BN1(self.CN1(x))))
        x = F.relu(self.MP_(self.BN2(self.CN2(x))))
        x = F.relu(self.MP_(self.BN3(self.CN3(x))))
        x = F.relu(self.MP_(self.BN4(self.CN4(x))))
        x = F.relu(self.MP_(self.BN5(self.CN5(x))))
        x = F.relu(self.MP_(self.BN6(self.CN6(x))))
        x = F.relu(self.MP_(self.BN7(self.CN7(x))))

        x = x.view(x.shape[0], -1)
        if self.use_DP:
            x = F.relu(self.DP_(self.FC1(x)))
        else:
            x = F.relu(self.FC1(x))
        x = self.FC2(x)
        return x




In [6]:
def predict_(
    model=None, dataloader=None, device=None, path=None, use_dropout=False, mc_iters=1
):
    """
    Load checkpoint if available, then run inference.
    If use_dropout=True, the model stays in train mode so dropout remains active.
    mc_iters defines how many Monte‑Carlo forward passes to average when dropout is on.
    """
    if path and os.path.isfile(path):
        model.load_state_dict(torch.load(path, map_location=device))
    else:
        print(f"[INFO] Checkpoint not found at '{path}'. Using untrained model.")
    model.to(device)

    if use_dropout:
        model.train()
    else:
        model.eval()

    accum = []  # store MC predictions
    for _ in range(mc_iters):
        batch_preds = []
        for batch in dataloader:
            X = batch.to(device)
            with torch.no_grad():
                pred = torch.sigmoid(model(X))
            batch_preds.append(pred.cpu())
        preds = torch.cat(batch_preds, dim=0)  # (N, 11)
        accum.append(preds.numpy())
    return np.mean(np.stack(accum, axis=0), axis=0)


def find_checkpoint(search_root="../input"):
    """
    Search for a .pt or .pth file in typical competition folders.
    Return the newest file (by modification time) if any are found.
    """
    patterns = [
        os.path.join(search_root, "**", "*.pt"),
        os.path.join(search_root, "**", "*.pth"),
    ]
    candidates = []
    for pat in patterns:
        candidates.extend(glob.glob(pat, recursive=True))
    if not candidates:
        return None
    # Choose the most recently modified file
    candidates.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    return candidates[0]


cfg = CFG(filter_sizes=[64, 128, 256, 512], HL=[2048], epochs=30, n_folds=5)

transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])]
)


def make_loader(img_array):
    ds = DS(X=img_array, y=None, transform=transform, mode="test")
    return DL(ds, batch_size=cfg.ts_batch_size, shuffle=False)


model = CNN(
    filter_sizes=cfg.filter_sizes,
    HL=cfg.HL,
    OL=cfg.OL,
    in_channels=cfg.in_channels,
    use_DP=True,
    DP=0.5,
)

# --- robust checkpoint handling ---
default_ckpt = "../input/rccl-384-train/Epoch_14.pt"
if os.path.isfile(default_ckpt):
    ckpt_path = default_ckpt
else:
    ckpt_path = find_checkpoint("../input")
    if ckpt_path:
        print(f"[INFO] Using discovered checkpoint at '{ckpt_path}'")
    else:
        print("[WARN] No checkpoint file found; inference will use untrained model.")
ckpt_exists = ckpt_path is not None and os.path.isfile(ckpt_path)

num_target_cols = ss.shape[1] - 1  # columns after the ID column

if ckpt_exists:
    loader_orig = make_loader(ts_images)
    pred_orig = predict_(
        model, loader_orig, cfg.device, ckpt_path, use_dropout=False, mc_iters=1
    )

    ts_hflip = np.flip(ts_images, axis=2).copy()
    loader_hflip = make_loader(ts_hflip)
    pred_hflip = predict_(
        model, loader_hflip, cfg.device, ckpt_path, use_dropout=False, mc_iters=1
    )

    ts_vflip = np.flip(ts_images, axis=1).copy()
    loader_vflip = make_loader(ts_vflip)
    pred_vflip = predict_(
        model, loader_vflip, cfg.device, ckpt_path, use_dropout=False, mc_iters=1
    )

    ts_both = np.flip(ts_hflip, axis=1).copy()
    loader_both = make_loader(ts_both)
    pred_both = predict_(
        model, loader_both, cfg.device, ckpt_path, use_dropout=False, mc_iters=1
    )

    ts_rot90 = np.stack(
        [
            TF.rotate(torch.from_numpy(img).permute(2, 0, 1), 90)
            .permute(1, 2, 0)
            .numpy()
            for img in ts_images
        ]
    )
    loader_rot90 = make_loader(ts_rot90)
    pred_rot90 = predict_(
        model, loader_rot90, cfg.device, ckpt_path, use_dropout=False, mc_iters=1
    )

    ts_rot270 = np.stack(
        [
            TF.rotate(torch.from_numpy(img).permute(2, 0, 1), 270)
            .permute(1, 2, 0)
            .numpy()
            for img in ts_images
        ]
    )
    loader_rot270 = make_loader(ts_rot270)
    pred_rot270 = predict_(
        model, loader_rot270, cfg.device, ckpt_path, use_dropout=False, mc_iters=1
    )

    pred_dropout = predict_(
        model, loader_orig, cfg.device, ckpt_path, use_dropout=True, mc_iters=3
    )

    y_pred = np.mean(
        np.stack(
            [
                pred_orig,
                pred_hflip,
                pred_vflip,
                pred_both,
                pred_rot90,
                pred_rot270,
                pred_dropout,
            ],
            axis=0,
        ),
        axis=0,
    )
else:
    train_path = "../input/ranzcr-clip-catheter-line-classification/train.csv"
    train_df = pd.read_csv(train_path)
    label_cols = train_df.columns[1:-1]  # drop ID and PatientID
    prevalence = train_df[label_cols].mean().values  # shape (num_target_cols,)
    y_pred = np.tile(prevalence, (ts_images.shape[0], 1))

if y_pred.shape[1] > num_target_cols:
    y_pred = y_pred[:, :num_target_cols]  # truncate excess logits
elif y_pred.shape[1] < num_target_cols:
    pad_width = num_target_cols - y_pred.shape[1]
    y_pred = np.concatenate([y_pred, np.zeros((y_pred.shape[0], pad_width))], axis=1)

y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)

ss.iloc[:, 1:] = y_pred
submission_path = "./submission.csv"
ss.to_csv(submission_path, index=False)
print(f"Submission saved to '{submission_path}'")
ss.head(5)

[WARN] No checkpoint file found; inference will use untrained model.
Submission saved to './submission.csv'


/tmp/ipykernel_55/624093868.py:171: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.00280712 0.00280712 0.00280712 ... 0.00280712 0.00280712 0.00280712]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ss.iloc[:, 1:] = y_pred
/tmp/ipykernel_55/624093868.py:171: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.03793307 0.03793307 0.03793307 ... 0.03793307 0.03793307 0.03793307]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ss.iloc[:, 1:] = y_pred
/tmp/ipykernel_55/624093868.py:171: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.24019354 0.24019354 0.24019354 ... 0.24019354 0.24019354 0.24019354]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ss.i

,StudyInstanceUID,ETT - Abnormal,ETT - Borderline,ETT - Normal,NGT - Abnormal,NGT - Borderline,NGT - Incompletely Imaged,NGT - Normal,CVC - Abnormal,CVC - Borderline
0,1.2.826.0.1.3680043.8.498.25512976433640891933...,0.002807,0.037933,0.240194,0.009382,0.017988,0.090604,0.159193,0.106929,0.281894
1,1.2.826.0.1.3680043.8.498.24449897997512078380...,0.002807,0.037933,0.240194,0.009382,0.017988,0.090604,0.159193,0.106929,0.281894
2,1.2.826.0.1.3680043.8.498.38485493636649999035...,0.002807,0.037933,0.240194,0.009382,0.017988,0.090604,0.159193,0.106929,0.281894
3,1.2.826.0.1.3680043.8.498.11901773728604504629...,0.002807,0.037933,0.240194,0.009382,0.017988,0.090604,0.159193,0.106929,0.281894
4,1.2.826.0.1.3680043.8.498.11327663178389439022...,0.002807,0.037933,0.240194,0.009382,0.017988,0.090604,0.159193,0.106929,0.281894
